# Preprocessing Data — Deteksi Sarkasme pada Dataset SARC

Notebook ini menyiapkan data agar siap dipakai untuk melatih **RoBERTa** dan **DistilBERT**. Semua langkah mengikuti **proposal** dan **temuan EDA** (01_eda.ipynb).

**Alur kerja:**


Data mentah (1.010.826 pasangan)
   → Cleaning teks (parent & comment)
   → Buang data bermasalah (kosong, label bertentangan, duplikat, artefak /s)
   → Ambil subset 200.000 pasangan (100.000 sarkas + 100.000 non-sarkas)
   → Split stratified 80 / 10 / 10 (train / validasi / test)
   → Simpan ke data/split/


**Input:** data/raw/train-balanced-sarcasm.csv
**Output:** `data/split/train.parquet`, `val.parquet`, `test.parquet`, `train_pilot.parquet`, dan `split_config.json`

## 1. Setup & Konfigurasi

Semua angka penting ditaruh di satu tempat agar mudah dicek dan diubah:

| Konfigurasi | Nilai | Keterangan |
|---|---|---|
| `SEED` | 42 | Supaya hasil pengambilan sampel dan split selalu sama setiap kali dijalankan |
| `N_PER_CLASS` | 100.000 | 100.000 sarkas + 100.000 non-sarkas = 200.000 pasangan (proposal) |
| `TRAIN / VAL / TEST` | 80% / 10% / 10% | Pembagian data sesuai proposal |
| `PILOT_SIZE` | 40.000 | Data kecil untuk uji coba awal (proposal: 20.000–50.000) |

In [ ]:
import re, html, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

SEED = 42
N_PER_CLASS = 100_000                     # 2 x 100.000 = 200.000 pasangan (proposal)
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.8, 0.1, 0.1
PILOT_SIZE = 40_000                       # pengujian awal 20.000–50.000 data (proposal)

ROOT = Path.cwd().parents[1]
RAW_PATH = ROOT / "data/raw/train-balanced-sarcasm.csv"
PROCESSED_DIR = ROOT / "data/processed"
SPLIT_DIR = ROOT / "data/split"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = {0: "Non-Sarkas", 1: "Sarkas"}
sns.set_theme(style="whitegrid")


## 2. Load Data

Data mentah dibaca dan setiap baris diberi `raw_idx`, yaitu nomor baris aslinya. Tujuannya agar nanti, saat menganalisis kesalahan model, kita bisa melacak setiap data kembali ke dataset asli.

In [16]:
df_raw = pd.read_csv(RAW_PATH)

df_raw["raw_idx"] = df_raw.index          # ID baris asli, untuk penelusuran saat analisis error
print(df_raw.shape)
df_raw.head()


(1010826, 11)


,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment,raw_idx
0,0,NC and NH.,Trumpbart,politics,2,-1,-1,2016-10,2016-10-16 23:55:23,"Yeah, I get that argument. At this point, I'd ...",0
1,0,You do know west teams play against west teams...,Shbshb906,nba,-4,-1,-1,2016-11,2016-11-01 00:24:10,The blazers and Mavericks (The wests 5 and 6 s...,1
2,0,"They were underdogs earlier today, but since G...",Creepeth,nfl,3,3,0,2016-09,2016-09-22 21:45:37,They're favored to win.,2
3,0,"This meme isn't funny none of the ""new york ni...",icebrotha,BlackPeopleTwitter,-8,-1,-1,2016-10,2016-10-18 21:03:47,deadass don't kill my buzz,3
4,0,I could use one of those tools.,cush2push,MaddenUltimateTeam,6,-1,-1,2016-12,2016-12-30 17:00:13,Yep can confirm I saw the tool they use for th...,4


#### Insight
- Dataset berisi **1.010.826 pasangan** komentar dengan 10 kolom asli ditambah `raw_idx`.
- Kolom yang penting untuk model hanya tiga: `parent_comment` (konteks), `comment` (komentar yang dinilai), dan `label` (1 = sarkas, 0 = bukan sarkas).

## 3. Fungsi Text Cleaning

Fungsi `clean_text` membersihkan teks dengan langkah-langkah dari proposal. Fungsi yang **sama** dipakai untuk `comment` dan `parent_comment` agar keduanya diperlakukan setara.

| No | Langkah | Contoh | Alasan |
|---|---|---|---|
| 1 | Ubah kode HTML menjadi karakter asli | `&gt;` → `>` | Kalau tidak, akan tersisa kata aneh seperti "gt" |
| 2 | Hapus URL | `https://reddit.com/...` → *(hilang)* | Link tidak punya makna bahasa |
| 3 | Hapus tag HTML | `<b>` → *(hilang)* | Sisa format, bukan isi komentar |
| 4 | Hapus mention | `@user`, `/u/user` → *(hilang)* | Nama pengguna tidak menentukan sarkas atau tidak |
| 5 | Hapus emoji | 😂 → *(hilang)* | Sesuai batasan penelitian (hanya teks) |
| 6 | Huruf kecil semua | `GREAT` → `great` | Menyeragamkan teks |
| 7 | Hapus tanda baca | `great!!!` → `great` | Sesuai proposal |
| 8 | Rapikan spasi | `oh   great` → `oh great` | Menghapus spasi berlebih |

Urutan ini penting: URL dan kode HTML **harus dihapus sebelum** tanda baca. Jika terbalik, URL akan terpecah menjadi potongan kata seperti "https www reddit com".

Stopword removal, stemming, dan lemmatization **tidak** dilakukan, karena kata seperti *"not"*, *"sure"*, dan *"yeah"* justru penting untuk mengenali sarkasme.

In [17]:
URL_RE = r"(?i)https?://\S+|www\.\S+"     # (?i): tidak peka huruf besar/kecil
HTML_TAG_RE = r"<[^>]+>"
MENTION_RE = r"(?:^|\s)(?:/?u/\w+|@\w+)"
EMOJI_RE = r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]"
ARTIFACT_RE = r"you (?:forgot|dropped|missed) (?:the|your|this|a)"   # balasan pengingat "/s"

def clean_text(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str).map(html.unescape)      # 1. decode HTML entity (&gt; -> >)
    s = s.str.replace(URL_RE, " ", regex=True)           # 2. hapus URL
    s = s.str.replace(HTML_TAG_RE, " ", regex=True)      # 3. hapus tag HTML
    s = s.str.replace(MENTION_RE, " ", regex=True)       # 4. hapus mention
    s = s.str.replace(EMOJI_RE, " ", regex=True)         # 5. hapus emoji
    s = s.str.lower()                                    # 6. lowercase
    s = s.str.replace(r"[^\w\s]", " ", regex=True)       # 7. hapus tanda baca
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()   # 8. rapikan spasi
    return s

# Uji cepat fungsi
clean_text(pd.Series([
    "Oh GREAT, another &gt; quote... https://reddit.com/r/x",
    "Thanks @someone and /u/other_user!!! 😂",
])).tolist()


['oh great another quote', 'thanks and']

#### Insight
- Uji cepat menunjukkan fungsi bekerja sesuai harapan: `"Oh GREAT, another &gt; quote... https://reddit.com/r/x"` menjadi `"oh great another quote"`.
- Pola URL memakai `(?i)` agar URL berhuruf kapital (misalnya `Https://Www...`) juga ikut terhapus. Masalah ini ditemukan saat sanity check (bagian 5).

## 4. Cleaning Data

Setelah teks dibersihkan, data yang bermasalah dibuang satu per satu. Setiap langkah dicatat agar jelas berapa data yang hilang dan alasannya.

In [18]:
log = []
def record(step, data):
    log.append({"Langkah": step, "Sisa baris": len(data)})

df = df_raw.copy()
record("Data mentah", df)

df = df.dropna(subset=["comment"])
record("Hapus comment kosong (NaN)", df)

df["comment_clean"] = clean_text(df["comment"])
df["parent_clean"] = clean_text(df["parent_comment"])
df = df[(df["comment_clean"] != "") & (df["parent_clean"] != "")]
record("Hapus teks kosong setelah cleaning", df)

conflict = df.groupby(["comment_clean", "parent_clean"])["label"].transform("nunique") > 1
df = df[~conflict]
record("Hapus pasangan berlabel bertentangan", df)

df = df.drop_duplicates(["comment_clean", "parent_clean"])
record("Hapus pasangan duplikat", df)

df = df[~df["comment_clean"].str.fullmatch(ARTIFACT_RE)]
record("Hapus artefak /s", df)

df = df.reset_index(drop=True)

preprocessing_log = pd.DataFrame(log)
preprocessing_log["Dibuang"] = (-preprocessing_log["Sisa baris"].diff()).fillna(0).astype(int)
display(preprocessing_log.style.hide(axis="index"))


Langkah,Sisa baris,Dibuang
Data mentah,1010826,0
Hapus comment kosong (NaN),1010771,55
Hapus teks kosong setelah cleaning,1009362,1409
Hapus pasangan berlabel bertentangan,1009220,142
Hapus pasangan duplikat,1008073,1147
Hapus artefak /s,1004248,3825


#### Insight

| Langkah | Dibuang | Penjelasan sederhana |
|---|---:|---|
| Comment kosong (NaN) | 55 | Komentar tidak ada isinya sejak awal |
| Teks kosong setelah cleaning | 1.409 | Isinya hanya link, emoji, atau tanda baca (misalnya `"?"` atau `":)"`), sehingga kosong setelah dibersihkan |
| Label bertentangan | 142 | Pasangan yang sama persis tetapi diberi label berbeda. Model akan bingung jika belajar dari data seperti ini |
| Pasangan duplikat | 1.147 | Pasangan parent–comment yang sama muncul lebih dari sekali, jadi cukup disimpan satu |
| Artefak `/s` | 3.825 | Balasan seperti *"you forgot the"* yang maksudnya "kamu lupa menulis /s". Ini bukan sarkasme, tetapi selalu berlabel sarkas, sehingga model bisa "curang" menghafalnya |

- Total data yang dibuang hanya **6.578 baris (0,65%)**. Tersisa **1.004.248 pasangan** yang bersih.
- Artinya kualitas data asli sudah cukup baik, dan cleaning tidak mengubah karakter dataset secara besar.

## 5. Sanity Check Hasil Cleaning

Pemeriksaan ulang untuk memastikan cleaning benar-benar berhasil. Jika masih ada masalah, `assert` akan menghentikan notebook dengan pesan error, sehingga data yang belum bersih tidak terbawa ke tahap berikutnya.

In [19]:
URL_LEFTOVER_RE = r"\bhttps? www\b|\bwww \w+ (?:com|org|net)\b|\bhttps? \w+ (?:com|org|net)\b"   # pola URL setelah tanda baca dihapus

def count_both(pattern):
    return df["comment_clean"].str.contains(pattern).sum() + df["parent_clean"].str.contains(pattern).sum()

checks = pd.Series({
    "Masih ada sisa URL": count_both(URL_LEFTOVER_RE),
    "Masih ada tanda baca": count_both(r"[^\w\s]"),
    "Masih ada huruf kapital": count_both(r"[A-Z]"),
    "Masih ada teks kosong": (df["comment_clean"] == "").sum() + (df["parent_clean"] == "").sum(),
    "Masih ada pasangan duplikat": df.duplicated(["comment_clean", "parent_clean"]).sum(),
}, name="Jumlah (harus 0)")
display(checks.to_frame())
assert checks.sum() == 0, "Masih ada data yang belum bersih!"

with pd.option_context("display.max_colwidth", 80):
    display(df[["comment", "comment_clean"]].sample(5, random_state=SEED))


,Jumlah (harus 0)
Masih ada sisa URL,0
Masih ada tanda baca,0
Masih ada huruf kapital,0
Masih ada teks kosong,0
Masih ada pasangan duplikat,0


,comment,comment_clean
417007,More of a syntha 6 type of guy,more of a syntha 6 type of guy
961120,lol isn't rape funny?,lol isn t rape funny
8469,But.....but....millenials are lazy kids who rely on credit cards and family ...,but but millenials are lazy kids who rely on credit cards and family to get ...
392014,Wan't there this weird Twilight Zone revival show on the air around late 90s...,wan t there this weird twilight zone revival show on the air around late 90s...
733376,It's just when you suggest deportation of a religious group for belonging to...,it s just when you suggest deportation of a religious group for belonging to...


#### Insight
- Semua pemeriksaan bernilai **0**: tidak ada lagi sisa URL, tanda baca, huruf kapital, teks kosong, atau pasangan duplikat.
- Contoh sebelum dan sesudah menunjukkan isi kalimat tetap utuh. Yang hilang hanya tanda baca dan huruf kapital, misalnya `"lol isn't rape funny?"` menjadi `"lol isn t rape funny"`.
- Catatan: kata "http" di dalam kalimat biasa (misalnya *"running a blog over HTTPS"*) tidak dianggap URL, karena itu memang bagian dari kalimat.

## 6. Subset 200.000 Pasangan

Sesuai proposal, eksperimen memakai sekitar **200.000 pasangan** yang dipilih secara terkontrol: diambil acak dengan seed tetap, **100.000 dari masing-masing label**.

In [20]:
subset = (
    df.groupby("label", group_keys=False)
      .sample(N_PER_CLASS, random_state=SEED)
      .reset_index(drop=True)
)
print(subset.shape)
subset["label"].value_counts()


(200000, 13)


label
0    100000
1    100000
Name: count, dtype: int64

#### Insight
- Subset berisi tepat **200.000 pasangan**: 100.000 sarkas dan 100.000 non-sarkas, sehingga seimbang sempurna.
- Jumlah ini sekitar 20% dari data bersih. Di EDA (bagian 7.2) sudah dibuktikan bahwa subset dengan seed yang sama punya distribusi subreddit, tahun, dan panjang teks yang hampir identik dengan data penuh, jadi subset ini **mewakili** keseluruhan data.

## 7. Split Stratified 80/10/10

Data dibagi menjadi tiga bagian:
- **Train (80%)**: untuk melatih model.
- **Validasi (10%)**: untuk memantau model selama training dan memilih hyperparameter.
- **Test (10%)**: untuk penilaian akhir. Data ini tidak boleh dilihat model sebelum evaluasi.

*Stratified* artinya proporsi sarkas dan non-sarkas dijaga sama di ketiga bagian. Split dilakukan dua kali: 80% train dan 20% sisa, lalu sisa dibagi dua sama rata menjadi validasi dan test.

In [21]:
train_df, temp_df = train_test_split(
    subset, train_size=TRAIN_SIZE, stratify=subset["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE), stratify=temp_df["label"], random_state=SEED
)
splits = {"train": train_df, "val": val_df, "test": test_df}

split_summary = pd.DataFrame({
    name: {
        "Jumlah": len(d),
        "% dari subset": len(d) / len(subset) * 100,
        "% Sarkas": d["label"].mean() * 100,
        "Median kata comment": d["comment_clean"].str.split().str.len().median(),
        "Median kata parent": d["parent_clean"].str.split().str.len().median(),
    }
    for name, d in splits.items()
}).T.round(2)
display(split_summary)


,Jumlah,% dari subset,% Sarkas,Median kata comment,Median kata parent
train,160000.0,80.0,50.0,9.0,14.0
val,20000.0,10.0,50.0,9.0,14.0
test,20000.0,10.0,50.0,9.0,14.0


#### Insight
- Hasil pembagian: **train 160.000**, **validasi 20.000**, **test 20.000**.
- Ketiganya berisi tepat **50% sarkas**, dan median panjang teksnya sama (comment 9 kata, parent 14 kata).
- Karena ketiga bagian sangat mirip, hasil evaluasi pada data test nanti akan adil dan bisa dipercaya.

## 8. Cek Kebocoran Data antar Split

*Data leakage* (kebocoran data) terjadi jika data yang sama ada di train dan di test. Model lalu terlihat pintar, padahal hanya menghafal. Ada tiga pemeriksaan:
- **Pasangan sama**: pasangan parent–comment yang identik di dua split (harus 0).
- **raw_idx sama**: baris asli yang sama di dua split (harus 0).
- **Parent sama**: parent yang sama tetapi dibalas oleh comment yang berbeda (hanya informasi).

In [22]:
def pair_keys(d):
    return set(zip(d["comment_clean"], d["parent_clean"]))

def overlap(col_fn):
    return {
        "train–val": len(col_fn(train_df) & col_fn(val_df)),
        "train–test": len(col_fn(train_df) & col_fn(test_df)),
        "val–test": len(col_fn(val_df) & col_fn(test_df)),
    }

leak = pd.DataFrame({
    "Pasangan sama (harus 0)": overlap(pair_keys),
    "raw_idx sama (harus 0)": overlap(lambda d: set(d["raw_idx"])),
    "Parent sama (info)": overlap(lambda d: set(d["parent_clean"])),
})
display(leak)
assert leak["Pasangan sama (harus 0)"].sum() == 0 and leak["raw_idx sama (harus 0)"].sum() == 0

shared_parents = set(train_df["parent_clean"]) & set(test_df["parent_clean"])


,Pasangan sama (harus 0),raw_idx sama (harus 0),Parent sama (info)
train–val,0,0,302
train–test,0,0,313
val–test,0,0,64


#### Insight
- **Tidak ada kebocoran**: pasangan sama dan `raw_idx` sama bernilai 0 di semua kombinasi split.
- Ada parent yang muncul di dua split: 302 (train–val), 313 (train–test), dan 64 (val–test). Ini **bukan kebocoran**, karena comment dan labelnya berbeda. Contohnya, satu postingan "how so" atau "seriously" bisa dibalas banyak orang dengan komentar yang berbeda.
- Jumlahnya kecil (sekitar 1,5% dari data test), sehingga tidak berpengaruh besar dan cukup dicatat sebagai informasi.

## 9. Data Pilot (Pengujian Awal)

Sesuai proposal, sebelum training penuh dilakukan uji coba dengan data yang lebih kecil untuk:
- memastikan seluruh pipeline (tokenisasi → training → evaluasi) berjalan tanpa error,
- menentukan *batch size* yang muat di memori GPU.

Data pilot diambil **dari data train saja**, agar data validasi dan test tetap "bersih".

In [23]:
pilot_df, _ = train_test_split(
    train_df, train_size=PILOT_SIZE, stratify=train_df["label"], random_state=SEED
)
print("Pilot:", pilot_df.shape, "| % sarkas:", round(pilot_df["label"].mean() * 100, 2))

Pilot: (40000, 13) | % sarkas: 50.0


#### Insight
- Data pilot berisi **40.000 pasangan** dengan proporsi sarkas tetap **50%**.
- Karena diambil dari train, pilot bisa langsung dievaluasi memakai data validasi yang sama dengan eksperimen utama.

## 10. Simpan Hasil

Setiap split disimpan dalam format **Parquet**, yang lebih cepat dibaca dan lebih kecil daripada CSV. Kolom yang disimpan:

| Kolom | Fungsi |
|---|---|
| `parent_clean` | **Input model**: kalimat pertama (konteks) |
| `comment_clean` | **Input model**: kalimat kedua (komentar yang dinilai) |
| `label` | **Target**: 1 = sarkas, 0 = non-sarkas |
| `raw_idx` | Nomor baris asli, untuk penelusuran |
| `parent_comment`, `comment` | Teks asli, untuk analisis kesalahan model |
| `subreddit`, `date` | Informasi tambahan, untuk analisis per kelompok (tidak dipakai saat training) |

Konfigurasi preprocessing juga disimpan ke `split_config.json`, sehingga siapa pun bisa mengulang proses ini dan mendapatkan hasil yang sama.

In [24]:
KEEP_COLS = ["raw_idx", "label", "parent_clean", "comment_clean",
             "parent_comment", "comment", "subreddit", "date"]

for name, d in {**splits, "train_pilot": pilot_df}.items():
    path = SPLIT_DIR / f"{name}.parquet"
    d[KEEP_COLS].reset_index(drop=True).to_parquet(path, index=False)
    print(f"{name:12s} {len(d):>7,} baris -> {path}")

preprocessing_log.to_csv(PROCESSED_DIR / "preprocessing_log.csv", index=False)
split_summary.to_csv(PROCESSED_DIR / "split_summary.csv")

config = {
    "seed": SEED,
    "n_per_class": N_PER_CLASS,
    "split_ratio": {"train": TRAIN_SIZE, "val": VAL_SIZE, "test": TEST_SIZE},
    "pilot_size": PILOT_SIZE,
    "n_after_cleaning": len(df),
    "n_rows": {name: len(d) for name, d in {**splits, "train_pilot": pilot_df}.items()},
    "text_columns": {"first": "parent_clean", "second": "comment_clean"},
    "preprocessing": ["html_unescape", "remove_url", "remove_html_tag", "remove_mention",
                      "remove_emoji", "lowercase", "remove_punctuation", "normalize_whitespace",
                      "drop_empty", "drop_conflicting_label", "drop_duplicate_pair", "drop_s_artifact"],
}
(SPLIT_DIR / "split_config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))


train        160,000 baris -> /Users/hibrizi/Project/TA/data/split/train.parquet
val           20,000 baris -> /Users/hibrizi/Project/TA/data/split/val.parquet
test          20,000 baris -> /Users/hibrizi/Project/TA/data/split/test.parquet
train_pilot   40,000 baris -> /Users/hibrizi/Project/TA/data/split/train_pilot.parquet
{
  "seed": 42,
  "n_per_class": 100000,
  "split_ratio": {
    "train": 0.8,
    "val": 0.1,
    "test": 0.1
  },
  "pilot_size": 40000,
  "n_after_cleaning": 1004248,
  "n_rows": {
    "train": 160000,
    "val": 20000,
    "test": 20000,
    "train_pilot": 40000
  },
  "text_columns": {
    "first": "parent_clean",
    "second": "comment_clean"
  },
  "preprocessing": [
    "html_unescape",
    "remove_url",
    "remove_html_tag",
    "remove_mention",
    "remove_emoji",
    "lowercase",
    "remove_punctuation",
    "normalize_whitespace",
    "drop_empty",
    "drop_conflicting_label",
    "drop_duplicate_pair",
    "drop_s_artifact"
  ]
}


## 11. Verifikasi File Tersimpan

Membaca kembali file yang sudah disimpan untuk memastikan isinya benar.

In [ ]:
check = pd.read_parquet(SPLIT_DIR / "train.parquet")
print(check.shape)
check.head()


#### Insight
- File `train.parquet` berhasil dibaca kembali: **160.000 baris dan 8 kolom**, sesuai dengan yang disimpan.

## 12. Kesimpulan & Langkah Selanjutnya

### Kesimpulan
1. **Cleaning berhasil dan aman.** Semua langkah proposal diterapkan pada parent dan comment. Hanya 0,65% data yang dibuang, dan sanity check memastikan tidak ada URL, tanda baca, huruf kapital, teks kosong, atau duplikat yang tersisa.
2. **Data bermasalah sudah dibuang**, yaitu label bertentangan dan artefak `/s`, sehingga model tidak belajar dari data yang membingungkan atau menyesatkan.
3. **Data eksperimen siap pakai.** Subset 200.000 pasangan dibagi menjadi 160.000 / 20.000 / 20.000 dengan proporsi label seimbang di setiap bagian.
4. **Tidak ada kebocoran data** antara train, validasi, dan test.
5. **Hasil bisa diulang.** Seed dan konfigurasi tersimpan, jadi proses ini selalu menghasilkan data yang sama.